# 10. Opt-In Poster Boards, HTML Export & Ecosystem Adapters

Everyday `xkcd-matchem` figures stay open and borderless on `#FFFFFF` paper. When you want to assemble a conference poster or lab handout, you can **opt into** `xm.Board`, `Container`, `Card`, and `Badge` using exact millimeter coordinates.

Every card owns a regular Matplotlib `Axes` (`card.axes`), and you can **stack multiple cards vertically or side by side inside container boxes** to organize poster sections cleanly.

In [ ]:
# Install xkcd-matchem automatically in Google Colab
import importlib.util


if importlib.util.find_spec("xkcd_matchem") is None:
    %pip install -q "git+https://github.com/vzordillo/xkcd-matchem.git"


## Basic Usage

Below we create two side-by-side `Container` boxes: on the left we **stack two cards vertically inside a single box**, and on the right we place a taller card with `header_band=True` alongside an open borderless card (`header_band=False`, the default):

In [ ]:
import xkcd_matchem as xm


board = xm.Board(
    240.0,
    145.0,
    dpi=150,
    title="Stacking Cards Inside Container Boxes",
    subtitle="Vertical stacks + side-by-side cards in exact mm coordinates",
    preset="poster",
)
# Left box with two vertically stacked cards
board.add_container(
    6.0, 24.0, 112.0, 115.0, title="Box 1: Stacked Cards", solid_header=True
)
c_top = board.add_card(
    10.0, 35.0, 104.0, 47.0, title="1. Workflow (Borderless)"
)
c_bot = board.add_card(
    10.0,
    86.0,
    104.0,
    48.0,
    title="2. Functional Gaps",
    header_band=True,
    inner_pad_mm=(18.0, 5.0, 12.5, 13.0),
)

# Right box with a single full-height card
board.add_container(
    122.0,
    24.0,
    112.0,
    115.0,
    title="Box 2: Crystal Structure",
    solid_header=True,
    header_color="#2A9D8F",
)
c_right = board.add_card(
    126.0,
    35.0,
    104.0,
    99.0,
    title="TiO2 (110) Slab",
    header_band=True,
    header_color="#2A9D8F",
    inner_pad_mm=(4.5, 4.5, 11.5, 4.5),
)

xm.plot_pipeline_diagram(
    [("Parse", "CIF"), ("Filter", "Hull"), ("Render", "xkcd")],
    ax=c_top.axes,
    preset="poster",
)
xm.plot_bar_comparison(
    ["PBE", "r2SCAN", "HSE06"],
    [0.62, 1.04, 1.15],
    ylabel="Gap (eV)",
    ax=c_bot.axes,
    preset="poster",
)
xm.plot_slab("tio2_slab.poscar", view="side", ax=c_right.axes, preset="poster")
board.add_badge(210.0, 14.0, "Poster API")
board.figure

## Advanced Usage

### 1. Share Interactive 3D Structures as a Single Offline HTML File

Want collaborators or students to spin your crystal without installing Python? `xm.export_interactive_html(...)` bundles the coordinates, styles, and embedded `xkcd Script` font into one self-contained `.html` file that opens anywhere:

In [ ]:
import pathlib
import tempfile


with tempfile.TemporaryDirectory() as tmpdir:
    html_out = xm.export_interactive_html(
        "tio2_slab.poscar",
        pathlib.Path(tmpdir) / "tio2_studio.html",
        style="polyhedra",
        polyhedra=("Ti", "O"),
        include_bundled_samples=False,
    )
    print("Exported standalone HTML size (bytes):", html_out.stat().st_size)

### 2. Bridging to ASE, pymatgen, RDKit, cclib & the CLI

When a project already lives inside **ASE**, **pymatgen**, **RDKit**, **cclib**, **MDAnalysis**, or **MDTraj**, install the matching extra (`pip install "xkcd-matchem[ase,pymatgen,rdkit,qm,md] @ git+https://github.com/vzordillo/xkcd-matchem.git"`, or `pip install -e ".[ase,pymatgen,rdkit,qm,md]"` from a local clone) and convert objects in one call:

```python
import importlib.util
import xkcd_matchem as xm

# Optional ecosystem bridges (available when extras are installed):
if importlib.util.find_spec("rdkit"):
    atoms = xm.from_smiles("CC(=O)Oc1ccccc1C(=O)O")  # RDKit 3D conformer
if importlib.util.find_spec("ase"):
    atoms = xm.io.read_via_ase(xm.get_sample_path("aspirin.xyz"))
if importlib.util.find_spec("pymatgen"):
    from pymatgen.core import Structure

    pmg_structure = Structure.from_file(xm.get_sample_path("si_diamond.cif"))
    atoms = xm.from_pymatgen(pmg_structure)
if importlib.util.find_spec("cclib"):
    cc_atoms = xm.io.read_via_cclib(xm.get_sample_path("h2o_gaussian.log"))
```

Prefer rendering straight from your terminal? Every structure camera, supercell, and custom color override works from the CLI too:

```bash
xkcd-matchem view mos2_2h.poscar --supercell 2 2 1 --view perspective --backend browser
xkcd-matchem render tio2_slab.poscar -o tio2.png --colors Ti=#E07A5F O=#264653
```

:::{tip}
**You're all set!** Save exact-millimeter PNG, SVG, or PDF posters via `board.save("my_poster.pdf", exact_mm=True)` or `xm.save_figure(fig, "figure.svg")`, and head to **Tutorials 11–14** for the xkcd Periodic Table, vintage parchment scrolls, background grid papers, hatched shading, matchem icons, GNN crystal graphs, 3D isosurfaces, phonons, Wulff shapes, comic strips, Brillouin zones, LOBSTER COHP, crystal-field splitting, electrochemistry, NMR/XPS, and turnkey SciComm templates!
:::